In [1]:

import fine.IOManagement.xarrayIO as xrIO

%load_ext autoreload
%autoreload 2

C:\Users\l.wijesinghe\fine\fine\subclasses\conversionDynamic.py:28: SyntaxWarning: invalid escape sequence '\]'
  """


# How to save an energy system model instance and set it back up? 

**Xarray and NetCDF files to the rescue!** The data contained within an Energy System Model (ESM) instance and the optimization results is vast and complex. Saving it directly is not possible. It can, however, be saved as a NetCDF file which supports complex data structures. 

#### What exactly is NetCDF? 
NetCDF (Network Common Data Format) is a set of software libraries and machine-independent data formats that support the creation, access, and sharing of array-oriented scientific data. It is also a community standard for sharing scientific data. 

#### Python modules that support working with NetCDF files:
1. netcdf4-python: Official Python interface to netCDF files
2. PyNIO: To access different file formats such as netCDF, HDF, and GRIB
3. xarray: Based on NumPy and pandas

Note: xarray module is used here. 

For our use case, the following functionalities are provided: 
* Conversion of ESM instance to xarray dataset. Additionally, possible to save this dataset as NetCDF file in a desired folder, with a desired file name. 
* Conversion of xarray dataset/saved NetCDF file back to ESM instance.

#### High-level structure of the data: 

<img src="overall_structure.png" style="width: 1000px;"/>


#### Structure of xarray dataset - For a non-transmission component: 

<img src="non_transmission.png" style="width: 1000px;"/>

#### Structure of xarray dataset - For a transmission component: 

<img src="transmission.png" style="width: 1000px;"/>


## Conversion of ESM instance to xarray dataset and saving it as a NetCDF file

### STEP 1. Set up your  ESM instance 

In [2]:
from getModel import getModel

esM = getModel()
esM.optimize()

Set parameter Username

--------------------------------------------
--------------------------------------------

Academic license - for non-commercial use only - expires 2025-06-18
Read LP format model from file C:\Users\LBDBA~1.WIJ\AppData\Local\Temp\tmp6y0pffh8.pyomo.lp
Reading time = 0.00 seconds
x1: 96 rows, 74 columns, 240 nonzeros
Set parameter QCPDual to value 1
Set parameter Threads to value 3
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11+.0 (26100.2))

CPU model: Intel(R) Core(TM) i7-8550U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 3 threads

Optimize a model with 96 rows, 74 columns and 240 nonzeros
Model fingerprint: 0xe19e9583
Coefficient statistics:
  Matrix range     [3e-01, 2e+03]
  Objective range  [7e-03, 9e+01]
  Bounds range     [1e+07, 2e+09]
  RHS range        [0e+00, 0e+00]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerica

### STEP 2. Conversion to xarray datasets and saving as NetCDF file
You can convert the esM to xarray datasets with `esm_to_datasets` and access Input, Parameters or Result.


In [3]:
esm_datasets = xrIO.writeEnergySystemModelToDatasets(esM)

In [4]:
esm_datasets["Input"]["Sink"]["Industry site"][
    "ts_operationRateFix"
].to_dataframe().unstack()

ts_operationRateFix                 
space ElectrolyzerLocation IndustryLocation
time                                       
0                      0.0       13140000.0
1                      0.0       13140000.0
2                      0.0       13140000.0
3                      0.0       13140000.0

In [5]:
esm_datasets["Results"][0]["SourceSinkModel"]["Electricity market"]

<xarray.Dataset> Size: 544B
Dimensions:                          (space: 2, time: 4)
Coordinates:
  * time                             (time) int64 32B 0 1 2 3
  * space                            (space) <U20 160B 'ElectrolyzerLocation'...
Data variables: (12/19)
    NPVcontribution                  (space) float64 16B 1.52e+06 0.0
    TAC                              (space) float64 16B 1.52e+06 0.0
    capacity                         (space) float64 16B nan nan
    capexCap                         (space) float64 16B nan nan
    capexIfBuilt                     (space) float64 16B nan nan
    commissioning                    (space) float64 16B nan nan
    ...                               ...
    operation_1                      (space) float64 16B 7.509e+07 nan
    opexCap                          (space) float64 16B nan nan
    opexIfBuilt                      (space) float64 16B nan nan
    opexOp                           (space) float64 16B 0.0 nan
    revenueLifetimeShorteningResale  (space) float64 16B nan nan
    operationVariablesOptimum        (time, space) float64 64B 1.877e+07 ... nan
Attributes: (12/18)
    NPVcontribution:                  [1 Euro]
    TAC:                              [1 Euro/a]
    capacity:                         [kW$_{el}$]
    capexCap:                         [1 Euro/a]
    capexIfBuilt:                     [1 Euro/a]
    commissioning:                    [kW$_{el}$]
    ...                               ...
    operation:                        [kW$_{el}$*h/a]
    operation_1:                      [kW$_{el}$*h]
    opexCap:                          [1 Euro/a]
    opexIfBuilt:                      [1 Euro/a]
    opexOp:                           [1 Euro/a]
    revenueLifetimeShorteningResale:  [1 Euro]

In [6]:
esm_datasets["Parameters"]

<xarray.Dataset> Size: 0B
Dimensions:  ()
Data variables:
    *empty*
Attributes: (12/15)
    locations:                  {'ElectrolyzerLocation', 'IndustryLocation'}
    commodities:                {'electricity', 'hydrogen'}
    commodityUnitsDict:         {'electricity': 'kW$_{el}$', 'hydrogen': 'kW$...
    numberOfTimeSteps:          4
    hoursPerTimeStep:           2190
    startYear:                  0
    ...                         ...
    costUnit:                   1 Euro
    lengthUnit:                 km
    verboseLogLevel:            1
    balanceLimit:               None
    pathwayBalanceLimit:        None
    annuityPerpetuity:          False

Or save it directly to NetCDF with `esm_to_netcdf`:

In [7]:
_ = xrIO.writeEnergySystemModelToNetCDF(
    esM, outputFilePath="my_esm.nc", overwriteExisting=True
)

### STEP 3. Load esM from NetCDF file or xarray datasets

You can load an esM from file with `netcdf_to_esm`.

In [8]:
esm_from_netcdf = xrIO.readNetCDFtoEnergySystemModel("my_esm.nc")

In [9]:
esm_from_netcdf.getComponentAttribute("Industry site", "operationRateFix")

space,ElectrolyzerLocation,IndustryLocation
time,,
0,0.0,13140000.0
1,0.0,13140000.0
2,0.0,13140000.0
3,0.0,13140000.0


Or from datasets with `datasets_to_esm`.

In [10]:
esm_from_datasets = xrIO.convertDatasetsToEnergySystemModel(esm_datasets)

In [11]:
esm_from_datasets.getComponentAttribute("Industry site", "operationRateFix")

space,ElectrolyzerLocation,IndustryLocation
time,,
0,0.0,13140000.0
1,0.0,13140000.0
2,0.0,13140000.0
3,0.0,13140000.0


In [12]:
esm_datasets["Results"][0]["SourceSinkModel"]["Electricity market"][
    "operationVariablesOptimum"
]

<xarray.DataArray 'operationVariablesOptimum' (time: 4, space: 2)> Size: 64B
array([[18771428.57142857,               nan],
       [37542857.14285714,               nan],
       [       0.        ,               nan],
       [18771428.57142857,               nan]])
Coordinates:
  * time     (time) int64 32B 0 1 2 3
  * space    (space) <U20 160B 'ElectrolyzerLocation' 'IndustryLocation'